# NeSy trên nền Neuron

Code: `zeroshot/Neuron/model/nesy.py` (mô hình), `kg.py` (đọc KG), `main_nesy.py` (train / eval), config trong `config/nesy/`.

- **Nhánh neural**: head của Neuron (micro-prototype spatial + temporal), giữ nguyên, train bằng loss của Neuron.
- **Nhánh symbolic**: 71 slot primitive có tên (query lấy từ CLIP text của primitive, attention chỉ trên các khớp của primitive, tách bên trái / phải) → giá trị chân lý `p_k(t)` → luật require / forbid / sequence trong `kg_rules.json` → `s_sym`.
- **Fusion**: `z = τ·s_neu + β·s_sym` (β học được); lúc test loại các lớp mâu thuẫn số người.

Cột kết quả (mỗi mẫu **một** dự đoán): `zsl_s_neu` (Neuron, cộng spatial + temporal), `zsl_s_sym`, `zsl_z` (fused), `zsl_final` (fused + mask số người). `zsl_neuron_or` là cách tính của paper Neuron (đúng nếu spatial HOẶC temporal đúng), chỉ để đối chiếu. GZSL: γ dò trên test như Neuron.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, subprocess
import pandas as pd

DRIVE_DIR   = '/content/drive/MyDrive/BaoHuy/zeroshot'
WEIGHTS_DIR = f'{DRIVE_DIR}/shiftgcn_weights'
SKEL = {'ntu60':  '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz',
        'ntu120': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU120_CSub.npz'}
OUT_DIR = f'{DRIVE_DIR}/nesy_neuron'          # work_dir trong config/nesy/*.yaml

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
NEURON = f'{REPO}/zeroshot/Neuron'
print('weights:', len(glob.glob(f'{WEIGHTS_DIR}/**/*.pt', recursive=True)), 'file')

In [ ]:
# Đọc npz trực tiếp từ Drive rất chậm: chép về đĩa local của Colab một lần.
def local_npz(ds):
    dst = f'/content/data/{os.path.basename(SKEL[ds])}'
    if not os.path.exists(dst):
        os.makedirs('/content/data', exist_ok=True)
        shutil.copy(SKEL[ds], dst)
    print(dst, f'{os.path.getsize(dst) / 2**30:.1f} GB')

local_npz('ntu60')

In [ ]:
# Chạy một split. Lần đầu: mã hóa text primitive bằng CLIP, rồi trích cube của encoder một lần
# (mỗi mẫu train một augment cố định). Bị ngắt thì chạy lại cell này: tiếp tục từ last.pt trong work_dir.
CFG = "ntu60_xsub_55_5"          # ntu60_xsub_55_5 | ntu60_xsub_48_12 | ntu120_xsub_110_10 | ntu120_xsub_96_24
if CFG.startswith("ntu120"):
    local_npz("ntu120")
!cd {NEURON} && python main_nesy.py --config config/nesy/{CFG}.yaml

In [ ]:
# Tổng hợp các split đã chạy: epoch cuối và best.pt (epoch có best_metric cao nhất trên test)
rows = []
for f in sorted(glob.glob(f"{OUT_DIR}/*/summary.json")):
    s = json.load(open(f))
    for name, r in (("last", s["last_epoch"]), (f"best.pt (ep {s['best_epoch']})", s["best_checkpoint"])):
        rows.append(dict(split=s["split"], row=name,
                         **{k: r[k] for k in ("zsl_s_neu", "zsl_s_sym", "zsl_z", "zsl_final", "zsl_neuron_or",
                                              "gzsl_s_neu_H", "gzsl_final_H")}))
display(pd.DataFrame(rows).round(4))

In [ ]:
# Đường học theo epoch của một split
m = pd.read_csv(f'{OUT_DIR}/{json.load(open(glob.glob(f"{OUT_DIR}/*/summary.json")[0]))["split"]}/metrics.csv')
cols = ['zsl_s_neu', 'zsl_s_sym', 'zsl_z', 'zsl_final', 'zsl_neuron_or']
ax = m.plot(x='epoch', y=cols, figsize=(9, 4), grid=True, title='ZSL theo epoch')
display(m[['epoch'] + cols + ['gzsl_final_H', 'beta', 'L_neuron', 'L_sym', 'L_fuse', 'L_logic']].round(4))